# 18 — Active Reset Benchmarking

## Purpose

Tune and calibrate the active-reset protocol that uses mid-circuit measurement and conditional π-pulses to rapidly initialize the qubit in |g⟩. Active reset replaces passive thermal relaxation, dramatically reducing the duty cycle for repeated experiments.

## Methodology

1. **Optimal readout / active-reset tuning** — sweep active-reset parameters (measurement threshold, number of rounds, conditional pulse type) to maximize reset fidelity
2. **Pulse-train calibration** — calibrate the multi-shot readout-and-reset pulse sequence that executes the reset protocol on hardware

## Expected Outcomes

- Reset fidelity > 99% (residual excited-state population < 1%)
- Calibrated pulse-train parameters persisted for use in all downstream experiments
- Cycle time reduction compared to passive thermal relaxation (typically 5–10× faster)

## Prerequisites

- **Notebook 16** — readout discrimination threshold calibrated (required for mid-circuit measurement)
- **Notebook 05** — π-pulse calibrated (conditional reset pulse)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the readout calibration checkpoint from notebook 16.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    PulseTrainCalibration,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="18_active_reset_benchmarking",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

readout_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if readout_checkpoint is not None:
    print(f"Prior stage 16 status: {readout_checkpoint['status']}")

## 2. Configuration — Active Reset Defaults

Set averaging counts and calibration-apply flags for each benchmarking step.

In [ ]:
APPLY_PULSE_TRAIN_CALIBRATION = True

# --- Optimal readout / active-reset tuning (Exp 48) ---
ACTIVE_RESET_N_AVG = 50000

# --- Pulse-train calibration (Exp 52) ---
PULSE_TRAIN_N_AVG = 10000

print("Active reset benchmarking settings:")
print(f"  apply pulse train calibration: {APPLY_PULSE_TRAIN_CALIBRATION}")
print(f"  active reset n_avg: {ACTIVE_RESET_N_AVG}")
print(f"  pulse train n_avg: {PULSE_TRAIN_N_AVG}")

## 3. Execution — Optimal Readout / Active Reset Tuning

Sweep active-reset parameters to maximize reset fidelity. The sweep explores measurement threshold, number of feedback rounds, and conditional pulse type.

In [ ]:
active_reset_result = session.optimal_readout_active_reset(
    n_avg=ACTIVE_RESET_N_AVG,
)

print("Active reset tuning complete.")

## 4. Execution — Pulse-Train Calibration

Calibrate the multi-shot readout-and-reset pulse sequence. The calibrated parameters are applied to the session if the fit passes the quality gate.

In [ ]:
pt_cal = PulseTrainCalibration(session)
pt_cal_result = pt_cal.run(n_avg=PULSE_TRAIN_N_AVG)
pt_cal_analysis = pt_cal.analyze(pt_cal_result, update_calibration=True)
pt_cal.plot(pt_cal_analysis)

pt_fit_ok = fit_quality_gate(pt_cal_analysis, r_squared_min=0.80)

if pt_fit_ok:
    pt_patch, pt_patch_preview, pt_apply_result = preview_or_apply_patch_ops(
        session,
        reason="Pulse-train calibration for active reset",
        proposed_patch_ops=pt_cal_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_PULSE_TRAIN_CALIBRATION,
    )
    if pt_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: Pulse-train calibration fit quality gate FAILED — skipping apply.")

print("Pulse-train calibration complete.")

## 5. Checkpoint — Save Active Reset Results

Persist the calibrated active-reset parameters. All downstream experiments can use the calibrated reset protocol for fast qubit initialization.

In [ ]:
pt_applied = "pt_apply_result" in dir() and pt_apply_result is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="18_active_reset_benchmarking",
    status="calibrated" if pt_applied else "characterized",
    summary="Active reset tuning and pulse-train calibration.",
    consumed_inputs={
        "active_reset_n_avg": ACTIVE_RESET_N_AVG,
        "pulse_train_n_avg": PULSE_TRAIN_N_AVG,
    },
    persisted_outputs={
        "pt_applied": pt_applied,
    },
    advisory_outputs={},
    next_stage="19_spa_optimization",
    notes=[
        "Active reset parameter sweep (Exp 48) uses session.optimal_readout_active_reset.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")